In [ ]:
import transformers
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_id = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

# Underthe hood uses tensors 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id).to(device)



In [ ]:
text = "Oh great, another broken screen. Just what I always wanted"

inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model(**inputs)
# logits
print(outputs)

# predicted class
predicted_class = outputs.logits.argmax(dim=-1).item()
print(model.config.id2label[predicted_class])


## Text Generating and Streaming

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM,  TextStreamer
import torch

# check gpu compatibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

# load tokenizer and model
model_id = "Qwen/Qwen3-0.6B" # MNN-Pocket-0.3B
tokenizer = AutoTokenizer.from_pretrained(model_id, return_tensors="pt")
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
streamer = TextStreamer(tokenizer=tokenizer, skip_prompt=True)

In [ ]:
# Text Generation
text = """
write a story on rich people
"""
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.inference_mode():
    # Probablistic sampling instead of greedy_sampling
    outputs = model.generate(
                                **inputs,
                                temperature=0.7,
                                do_sample=True,
                                max_new_tokens=200,
                                top_p=0.9,
                                repetition_penalty=1.4,
                            )
res = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(res) 

In [ ]:
# Streaming
prompt = "what is 5 plu 8."
inputs = tokenizer(
    prompt,
    return_tensors = "pt"
)

with torch.inference_mode():
    model.generate(
        **inputs,
        top_p=0.9,
        repetition_penalty=1.3,
        temperature=.7,
        max_new_tokens=500,
        do_sample=True,
        streamer=streamer,
        eos_token_id=tokenizer.eos_token_id
    )

In [ ]:
# Embeddings 
from sentence_transformers import SentenceTransformer

model_id = "Qwen/Qwen3-Embedding-0.6B"
embedding_model = SentenceTransformer(
    model_id,
    device=device
)

d:\PROJECTS\ProjectVision\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vivek\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\PROJECTS\ProjectVision\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingfa

tensor([0.7636, 0.1410])
tensor([0.1353, 0.5996])


In [18]:
queries = [
    "place"
]
documents = [
    "The capital of China is Beijing.",
    "Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.",
]

query_embedding = embedding_model.encode(queries, prompt_name="query")
document_embedding = embedding_model.encode(documents)

similarity = embedding_model.similarity(query_embedding, document_embedding)

ranked = sorted(zip(similarity[0].tolist() , documents), reverse=True)
for score, doc in ranked:
    print(f"[Score: {score:.4f}] {doc}")

[Score: 0.2660] The capital of China is Beijing.
[Score: 0.2444] Gravity is a force that attracts two bodies towards each other. It gives weight to physical objects and is responsible for the movement of planets around the sun.
